# Spatio-Temporal EEG Transformer (ST-EEGFormer)

This notebook trains and evaluates a Spatio-Temporal EEG Transformer (ST-EEGFormer) using pretrained encoder weights and LoRA adapters to decode binocular rivalry states (Incongruent Mirrored vs. Incongruent Pure).

### 📋 Spatio-Temporal EEG Transformer (ST-EEGFormer) Pipeline Details
* **Input Data**: Raw EEG epochs in `.fif` format.
* **Preprocessing & Feature Extraction**:
  * **Class Balancing**: Random undersampling per subject.
  * **Resampling & Filtering**: All 60 electrodes; resampled to 128 Hz; full window (0.0s to 1.5s); IIR bandpassed (1.0–40.0 Hz).
  * **Channel Alignment**: Rearranges the 60 electrode channels to match the 1D sequential spatial layout expected by ST-EEGFormer.
  * **Standardization**: Per-trial and per-channel Z-score scaling.
* **Model Architecture**:
  * Pre-trained ST-EEGFormer backbone.
  * Low-Rank Adaptation (LoRA) injected into the attention layers to allow parameter-efficient transfer learning with small sample sizes ($r=8$, $\alpha=16$).
  * Linear classifier head.
* **Training & Refitting**: AdamW optimizer ($lr=1e-4$), 2-stage automated refitting.
* **Evaluation**: 5-fold temporal stratified cross-validation.


### Spatio-Temporal EEG Transformer (ST-EEGFormer) Pipeline with LoRA Adapters

This model uses a ViT-style transformer block to process spatial and temporal EEG patches. Parameter-efficient fine-tuning is implemented via LoRA (Low-Rank Adaptation) adapters injected in Attention projection layers, while keeping the base ViT weights frozen.

```mermaid
graph TD
    A[Input Epochs: batch, channels, times] --> B[Reshape to batch*channels, 1, times]
    B --> C[Temporal Patch Embedding: 1D CNN]
    C --> D[Add Spatial & Temporal Positional Embeddings]
    D --> E[Flatten channels & patches to sequence]
    E --> F[Prepend Classification Token: cls_token]
    F --> G[Transformer Encoder Blocks with LoRA adapters]
    G --> H[Extract cls_token state & LayerNorm]
    H --> I[Fully Connected Classifier Head]
    I --> J[Classification Output: Incongruent Mirrored vs Incongruent Pure]
```

In [ ]:
# Unified Environment Setup (Local, Colab, Kaggle)
import os
import sys
from pathlib import Path

IS_KAGGLE = os.path.exists('/kaggle/working')
IS_COLAB = 'google.colab' in sys.modules

if IS_KAGGLE:
    print('Detected Kaggle environment. Installing dependencies...')
    !pip install -q mne braindecode[hub] safetensors timm shap huggingface_hub
elif IS_COLAB:
    print('Detected Google Colab environment. Installing dependencies...')
    !pip install -q mne braindecode[hub] safetensors timm shap huggingface_hub
    from google.colab import drive
    drive.mount('/content/drive')

# Locate workspace source directory
try:
    NOTEBOOK_DIR = Path(__file__).resolve().parent
except NameError:
    NOTEBOOK_DIR = Path.cwd()

if IS_KAGGLE:
    # Dynamically find the unzipped dataset path
    dataset_root = None
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'tfg-nils-2026-dataset.zip' in files or 'br_analysis' in dirs:
            dataset_root = Path(root)
            break
    SRC_DIR = dataset_root if dataset_root else Path('/kaggle/input/tfg-nils-2026-dataset')
elif IS_COLAB:
    SRC_DIR = Path('/content/drive/MyDrive/TFG_Nils_2026/03_Codi')
else:
    SRC_DIR = NOTEBOOK_DIR.parent

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f'Using source directory: {SRC_DIR}')


## Task And Environment Setup

In [ ]:
try:
    from IPython import get_ipython

    ipy = get_ipython()
    if ipy is not None:
        ipy.run_line_magic("load_ext", "autoreload")
        ipy.run_line_magic("autoreload", "2")
except Exception:
    pass

import numpy as np
import pandas as pd
import mne
import torch
from br_analysis import (
    detect_project_paths,
    get_task_config,
    load_raw_epoch_dataset,
    RawEpochSpec,
    get_epochs_grid,
    BinaryTaskConfig,
    COMMON_BAND_RANGES,
    TARGET_BANDS_CLUSTERS,
    temporal_band_features,
    electrode_distance_adjacency,
    load_fused_band_dataset,
    FusedBandSpec,
    temporal_stratified_splits,
)
from br_analysis.torch_training import (
    TorchTrainConfig,
    cross_validate_torch_model,
    standardize_train_test,
    build_dataloader,
    _evaluate,
)
from br_analysis.torch_models import (
    STGNNChebConv,
    EEGNet,
)
from br_analysis.pretrained import (
    prepare_lora_backbone,
    load_backbone_weights,
    BackboneClassifier,
    align_channels_to_steegformer,
)
from br_analysis import mae_vit_small_patch16
from braindecode.models import InterpolatedLaBraM

TASK_TAG = "im_vs_ip_steegformer"
paths = detect_project_paths("im_vs_i_steegformer")
task_name = "c_vs_ip" if "c_vs_ip" in TASK_TAG else "im_vs_ip"
task = get_task_config(task_name, paths.analysis_root)

mne.set_log_level("WARNING")
RANDOM_SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Working Paths:")
print("  Cache:", paths.cache_dir)
print("  Tables:", paths.tables_dir)
print("  Figures:", paths.figures_dir)

## Load Epochs And Align Channels

In [3]:
spec = RawEpochSpec(picks=None, bandpass=(1.0, 40.0), resample_sfreq=128.0)
dataset = load_raw_epoch_dataset(paths, task, spec)
X_raw = np.asarray(dataset.X, dtype=np.float32)
y = np.asarray(dataset.y, dtype=np.int64)
subjects = np.asarray(dataset.subjects, dtype=object)
SUBJECT_ID = "P057"
ch_names = list(dataset.channel_names)
times = dataset.times
sfreq = 128.0

# Use native epoch length
TARGET_LEN = X_raw.shape[-1]
X_raw = X_raw

# Align channels to standard 142 channels
X = align_channels_to_steegformer(X_raw, ch_names)

print(f"Aligned X shape: {X.shape} (trials, padded channels, times)")
print(f"Classes: {np.bincount(y)}")

Loading preprocessed dataset from cache: H:\.shortcut-targets-by-id\19KyEKxqClpX2n1-oU8TXRrmUspaPJKNU\TFG_Nils_2026\04_Anàlisi\cache\dataset_c_vs_ip_subj24_bp1.0_40.0_sf128.0_win0.0_1.5_pkall.joblib
Aligned 60/60 channels to 142-channel format.
Aligned X shape: (5870, 142, 192) (trials, padded channels, times)
Classes: [2935 2935]


## ST-EEGFormer Model Definition & Configuration

In [ ]:
def make_model():
    backbone = mae_vit_small_patch16()
    from huggingface_hub import hf_hub_download

    checkpoint_path = None
    if IS_KAGGLE:
        for root, dirs, files in os.walk("/kaggle/input"):
            if "ST-EEGFormer_small_encoder.safetensors" in files:
                checkpoint_path = Path(root) / "ST-EEGFormer_small_encoder.safetensors"
                break
    if checkpoint_path is None:
        checkpoint_path = hf_hub_download(
            repo_id="eugenehp/ST-EEGFormer",
            filename="ST-EEGFormer_small_encoder.safetensors",
        )
    load_backbone_weights(backbone, Path(checkpoint_path), strict=False)
    prepare_lora_backbone(
        backbone,
        include_substrings=("attn", "mlp"),
        exclude_substrings=("head", "classifier", "decoder", "patch_embed", "tokenizer"),
        r=8,
        alpha=16.0,
        dropout=0.05,
        train_classifier=False,
    )
    model = BackboneClassifier(backbone, n_classes=2, example_input=torch.from_numpy(X[:1]))
    return model


train_cfg = TorchTrainConfig(
    epochs=50,
    batch_size=8,
    lr=2e-4,
    weight_decay=1e-4,
    patience=15,
    n_splits=5,
    val_size=0.2,
    random_seed=RANDOM_SEED,
    device=DEVICE,
    progress=True,
)

## 3. Cohort-Wide Training (LoRA Fine-tuning)

We train the LoRA fine-tuned ST-EEGFormer model across all 24 subjects individually and cache the results.

In [ ]:
cohort_metrics_path = paths.tables_dir / f"cohort_cv_metrics_{TASK_TAG}.csv"
cohort_fold_metrics_path = paths.tables_dir / f"cohort_fold_metrics_{TASK_TAG}.csv"
subject_ids = pd.unique(subjects).tolist()

if cohort_metrics_path.exists() and cohort_fold_metrics_path.exists():
    print("Loading ST-EEGFormer cohort metrics from cache...")
    df_cohort = pd.read_csv(cohort_metrics_path)
    df_cohort_folds = pd.read_csv(cohort_fold_metrics_path)
else:
    print("Running ST-EEGFormer cohort-wide cross-validation with LoRA adapters...")
    cohort_results = []
    cohort_fold_results = []

    silent_cfg = TorchTrainConfig(
        epochs=train_cfg.epochs,
        batch_size=train_cfg.batch_size,
        lr=train_cfg.lr,
        weight_decay=train_cfg.weight_decay,
        patience=train_cfg.patience,
        n_splits=train_cfg.n_splits,
        val_size=train_cfg.val_size,
        random_seed=train_cfg.random_seed,
        device=train_cfg.device,
        progress=True,
    )

    for sub in subject_ids:
        subj_metrics_path = paths.cache_dir / f"metrics_{TASK_TAG}_sub{sub}.csv"
        if subj_metrics_path.exists():
            print(f"Loading cached metrics for subject {sub}...")
            steeg_metrics = pd.read_csv(subj_metrics_path)
        else:
            print(f"Training/Evaluating subject {sub}...")
            import gc

            gc.collect()
            torch.cuda.empty_cache()
            steeg_metrics, _ = cross_validate_torch_model(
                X,
                y,
                make_model,
                silent_cfg,
                subjects=subjects,
                subject_id=sub,
                save_weights_dir=paths.cache_dir,
                weights_prefix=f"{TASK_TAG}_sub{sub}",
                load_weights=True,
            )
            steeg_metrics.to_csv(subj_metrics_path, index=False)

        cohort_results.append(
            {
                "subject": sub,
                "steegformer_acc": steeg_metrics["test_acc"].mean(),
                "steegformer_auc": steeg_metrics["test_auc"].mean(),
                "steegformer_bal_acc": steeg_metrics["test_balanced_acc"].mean(),
            }
        )

        for idx, row in steeg_metrics.iterrows():
            cohort_fold_results.append(
                {
                    "subject": sub,
                    "model": "ST-EEGFormer",
                    "fold": int(row["fold"]),
                    "test_acc": row["test_acc"],
                    "test_auc": row["test_auc"],
                    "test_balanced_acc": row["test_balanced_acc"],
                }
            )

    df_cohort = pd.DataFrame(cohort_results)
    df_cohort.to_csv(cohort_metrics_path, index=False)
    df_cohort_folds = pd.DataFrame(cohort_fold_results)
    df_cohort_folds.to_csv(cohort_fold_metrics_path, index=False)
    print(f"Saved cohort metrics to: {cohort_metrics_path}")


# Global Cohort CV Summary (mean ± std)
print("\n" + "=" * 50)
print("GLOBAL COHORT CROSS-VALIDATION SUMMARY")
print("=" * 50)
for model_name in df_cohort_folds["model"].unique():
    model_df = df_cohort_folds[df_cohort_folds["model"] == model_name]
    print(f"Model: {model_name}")
    for f in sorted(model_df["fold"].unique()):
        fold_df = model_df[model_df["fold"] == f]
        mean_f = fold_df["test_acc"].mean()
        std_f = fold_df["test_acc"].std()
        print(f"  Fold {f} Accuracy: {mean_f:.4f} ± {std_f:.4f}")
    mean_cv = model_df["test_acc"].mean()
    std_cv = model_df["test_acc"].std()
    print(f"  Average CV Accuracy: {mean_cv:.4f} ± {std_cv:.4f}")
print("=" * 50 + "\n")

Loading ST-EEGFormer cohort metrics from cache...


## 3.5 Cohort-Wide Predictions Extraction

We extract model predictions across all subjects and folds once, caching them in memory so we don't have to reload models in subsequent sections.

In [ ]:
# Extract predictions across all subjects and folds
all_y_true = []
all_probs = []
subject_y_trues = {}
subject_y_probs = {}
subject_fold_predictions = {}  # {sub: {fold: (y_true, y_prob)}}

device = torch.device(DEVICE)

print("Extracting cohort predictions across all subjects and folds...")
for sub in subject_ids:
    splits = list(temporal_stratified_splits(y[subjects == sub], train_cfg.n_splits))
    X_sub = X[subjects == sub]
    y_sub = y[subjects == sub]

    sub_y_true = []
    sub_y_prob = []
    subject_fold_predictions[sub] = {}

    for fold, (train_idx, test_idx) in enumerate(splits, start=1):
        _, X_test = standardize_train_test(X_sub[train_idx], X_sub[test_idx])
        y_test = y_sub[test_idx]

        weights_path = paths.cache_dir / f"{TASK_TAG}_sub{sub}_fold{fold}.pt"
        if weights_path.exists():
            model = make_model().to(device)
            model.load_state_dict(torch.load(weights_path, map_location=device), strict=False)
            model.eval()

            loader = build_dataloader(X_test, y_test, batch_size=train_cfg.batch_size, shuffle=False, device=device)
            _, _, y_true_arr, p = _evaluate(model, loader, device)
            all_y_true.extend(y_true_arr)
            all_probs.extend(p)
            sub_y_true.extend(y_true_arr)
            sub_y_prob.extend(p)
            subject_fold_predictions[sub][fold] = (y_true_arr, p)

    if len(sub_y_true) > 0:
        subject_y_trues[sub] = np.array(sub_y_true)
        subject_y_probs[sub] = np.array(sub_y_prob)

all_y_true = np.array(all_y_true)
all_probs = np.array(all_probs)
all_preds = (all_probs >= 0.5).astype(int)

# ── Save Out-of-Fold predictions to NPZ file ──────────────────────────────────
oof_save_dict = {}
for sub in subject_ids:
    if sub in subject_y_trues and sub in subject_y_probs:
        oof_save_dict[f"{sub}_y_true"] = subject_y_trues[sub]
        oof_save_dict[f"{sub}_prob"] = subject_y_probs[sub]

if oof_save_dict:
    task_name_lc = "c_vs_ip" if "c_vs_ip" in TASK_TAG else "im_vs_ip"
    model_name_lc = "steegformer"
    comp_paths = detect_project_paths("cross_model_comparison")
    out_path = comp_paths.cache_dir / f"oof_predictions_{task_name_lc}_{model_name_lc}.npz"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(out_path, **oof_save_dict)
    print(f"Saved out-of-fold predictions to: {out_path}")

## 4. Single-Subject Detailed Performance Comparison (P057)

In [ ]:
from sklearn.metrics import classification_report, roc_curve, precision_recall_curve, auc
import matplotlib.pyplot as plt

# Extract P057-specific predictions from the cohort predictions
y_true_all = subject_y_trues[SUBJECT_ID]
probs = subject_y_probs[SUBJECT_ID]
preds = (probs >= 0.5).astype(int)

print("=== ST-EEGFormer Classification Report (Subject P057) ===")
print(classification_report(y_true_all, preds, target_names=[task.class_0, task.class_1], zero_division=0.0))

# Plot ROC/PR curves
fig, (ax_roc, ax_pr) = plt.subplots(1, 2, figsize=(11, 4.5))
fpr, tpr, _ = roc_curve(y_true_all, probs)
ax_roc.plot(fpr, tpr, label=f"ST-EEGFormer (AUC = {auc(fpr, tpr):.3f})", color="blue", linewidth=2)
ax_roc.plot([0, 1], [0, 1], "k--", alpha=0.5)
ax_roc.set_title("ROC Curve (Subject P057)")
ax_roc.set_xlabel("False Positive Rate")
ax_roc.set_ylabel("True Positive Rate")
ax_roc.legend()

prec, rec, _ = precision_recall_curve(y_true_all, probs)
ax_pr.plot(rec, prec, label=f"ST-EEGFormer (μP = {auc(rec, prec):.3f})", color="blue", linewidth=2)
ax_pr.plot([0, 1], [0.5, 0.5], "k--", alpha=0.5)
ax_pr.set_title("Precision-Recall Curve (Subject P057)")
ax_pr.set_xlabel("Recall")
ax_pr.set_ylabel("Precision")
ax_pr.legend()

plt.tight_layout()
plt.savefig(paths.figures_dir / f"roc_pr_{TASK_TAG}_P057.png", dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
# Zipping outputs for easy download when running on Kaggle
if IS_KAGGLE:
    import shutil

    print("Zipping outputs...")
    shutil.make_archive("/kaggle/working/outputs", "zip", "/kaggle/working/TFG_Nils_2026/04_Anàlisi")
    print("Zipped outputs saved to /kaggle/working/outputs.zip")